# Generative Models: Hands-on Notebook for Beginners

This notebook goes with the slide deck **"Generative Models: A Beginner's Guide"**. Every model from the slides is built and trained here, step by step:

| Part | Model | What you will see |
|---|---|---|
| 1 | The core idea | Learn the pattern of student heights, then sample new ones |
| 2 | Autoregressive models | Predict the next word; a tiny neural text model; GPT-2 |
| 3 | Autoencoder | Squeeze digits into 2 numbers and rebuild them |
| 4 | VAE | A tidy latent map: generate and morph digits |
| 5 | GAN | Forger vs. detective, first on a toy ring, then on digits |
| 6 | Diffusion | Add noise, learn to remove it, generate digits from static |
| 7 | Bonus | Text-to-image with Stable Diffusion (needs a GPU) |

**How to run it**
1. Open in **Google Colab** (File → Upload notebook).
2. Turn on a GPU: **Runtime → Change runtime type → T4 GPU**. Everything also runs on a CPU, just slower (the diffusion part is the slowest).
3. Run the cells from top to bottom with **Shift + Enter**.

Each section is independent after the *Setup* and *Load the data* cells, so you can jump to any model.

## 0. Setup
We use **PyTorch** for neural networks, **NumPy** for numbers and **Matplotlib** for pictures. Colab already has them installed.

`QUICK = True` makes every training loop tiny, which is useful for checking that everything runs in a couple of minutes. Keep it `False` for good-looking results.

In [ ]:
import math, random, time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

QUICK = False   # True = tiny training runs, just to test the notebook

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on:", device)

def set_seed(s=42):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
set_seed()

def show_grid(images, nrow=8, title=None, size=1.1):
    # Show a batch of 1x28x28 images (values in [0,1]) as a grid.
    images = images.detach().cpu().clamp(0, 1)
    n = images.shape[0]; ncol = nrow; nr = math.ceil(n / ncol)
    fig, axes = plt.subplots(nr, ncol, figsize=(ncol * size, nr * size))
    axes = np.array(axes).reshape(nr, ncol)
    for i, ax in enumerate(axes.flat):
        ax.axis("off")
        if i < n: ax.imshow(images[i, 0], cmap="gray")
    if title: fig.suptitle(title)
    plt.tight_layout(); plt.show()

---
## Part 1. The core idea: learn the pattern, then sample
*(Slides 5–6)*

Every generative model does three things:
1. **Collect data**: here, the heights of 400 students.
2. **Learn the pattern** (the *probability distribution*): for heights this is a bell curve with a mean and a spread.
3. **Sample**: draw brand-new values from that pattern.

In [ ]:
rng = np.random.default_rng(0)
heights = rng.normal(165, 8, 400)              # 1) our "real" data

mu, sigma = heights.mean(), heights.std()      # 2) learning = estimating the bell curve
print(f"Learned pattern: mean = {mu:.1f} cm, spread = {sigma:.1f} cm")

new_heights = rng.normal(mu, sigma, 8)         # 3) sampling NEW heights
print("New generated heights:", np.round(new_heights, 1))

x = np.linspace(135, 195, 300)
pdf = np.exp(-(x - mu)**2 / (2 * sigma**2)) / (sigma * np.sqrt(2 * np.pi))
plt.figure(figsize=(7, 3.5))
plt.hist(heights, bins=25, density=True, alpha=0.35, label="Real data")
plt.plot(x, pdf, lw=3, label="Learned model p(x)")
plt.scatter(new_heights, np.full(8, 0.002), marker="*", s=200, c="red", zorder=5, label="New samples")
plt.xlabel("Height (cm)"); plt.legend(); plt.show()

**Try it:** change `400` to `10`. With very little data the learned curve is less accurate, so samples are less trustworthy. Real generative models need *lots* of data for the same reason.

Images work the same way, except one image is 784 numbers (28 × 28 pixels) that depend on each other. A bell curve is far too simple for that, so from here on we use **neural networks** to learn the pattern.

---
## Part 2. Autoregressive models: predict the next piece
*(Slides 11–12, 25)*

An autoregressive model writes **one piece at a time**. At each step it asks: *"given everything so far, what comes next?"*

### 2a. Counting words: the simplest language model
We read a tiny story and count which word follows each pair of words. Those counts become probabilities.

In [ ]:
from collections import Counter, defaultdict

corpus = '''
the cat sat on the mat . the dog sat on the floor . the cat slept on the sofa .
the cat sat on the mat and the dog sat on the mat . a bird sat on the roof .
the cat jumped on the sofa . the dog slept on the floor . the cat sat on the mat .
the kid sat on the floor . the cat played on the keyboard . the dog sat on the sofa .
'''.split()

# count: (word1, word2) -> Counter of next words
counts = defaultdict(Counter)
for w1, w2, w3 in zip(corpus, corpus[1:], corpus[2:]):
    counts[(w1, w2)][w3] += 1

def next_word_probs(w1, w2):
    c = counts[(w1, w2)]
    total = sum(c.values())
    return {w: n / total for w, n in c.most_common()}

probs = next_word_probs("on", "the")
print('P(next word | "... on the") =')
for w, p in probs.items(): print(f"  {w:10s} {p:.0%}")

plt.figure(figsize=(6, 3))
plt.barh(list(probs)[::-1], list(probs.values())[::-1])
plt.title('The model\'s guess after "The cat sat on the ___"'); plt.xlabel("probability"); plt.show()

### The generation loop (slide 12) and the temperature knob
1. Start with a prompt → 2. get probabilities → 3. **sample** one word → 4. append it → repeat.

**Temperature** reshapes the probabilities before sampling:
* low (e.g. 0.2) → nearly always the most likely word (safe, repetitive)
* high (e.g. 2.0) → unlikely words get a real chance (creative, can be silly)

In [ ]:
def sample_with_temperature(prob_dict, temperature=1.0):
    words = list(prob_dict); p = np.array(list(prob_dict.values()))
    logits = np.log(p) / temperature             # temperature scaling
    p = np.exp(logits - logits.max()); p /= p.sum()
    return np.random.choice(words, p=p)

def generate(prompt, n_words=8, temperature=1.0, verbose=False):
    words = prompt.split()
    for step in range(n_words):
        probs = next_word_probs(words[-2], words[-1])
        if not probs: break                       # never seen this context
        words.append(sample_with_temperature(probs, temperature))
        if verbose: print(f"Step {step+1}: {' '.join(words)}")
        if words[-1] == ".": break
    return " ".join(words)

np.random.seed(1)
generate("the cat", verbose=True)
print()
for t in [0.2, 1.0, 2.0]:
    print(f"temperature {t}:", [generate("the cat", temperature=t) for _ in range(3)])

### 2b. A tiny *neural* autoregressive model
Counting only works for contexts we have seen before. A neural network **generalises**: it learns a representation of the text so far. Here a small recurrent network (GRU) learns to predict the next **character**. This is the same idea as GPT, just millions of times smaller.

In [ ]:
text = (" ".join(corpus) + " ") * 20
chars = sorted(set(text)); vocab = len(chars)
stoi = {c: i for i, c in enumerate(chars)}; itos = {i: c for c, i in stoi.items()}
data = torch.tensor([stoi[c] for c in text])
print("vocabulary size:", vocab, "| training characters:", len(data))

class CharModel(nn.Module):
    def __init__(self, vocab, emb=32, hidden=128):
        super().__init__()
        self.emb = nn.Embedding(vocab, emb)
        self.rnn = nn.GRU(emb, hidden, batch_first=True)
        self.head = nn.Linear(hidden, vocab)       # a score for every possible next character
    def forward(self, x, h=None):
        out, h = self.rnn(self.emb(x), h)
        return self.head(out), h

set_seed()
char_model = CharModel(vocab).to(device)
opt = torch.optim.Adam(char_model.parameters(), lr=3e-3)
seq_len, steps = 40, (150 if QUICK else 800)
for step in range(steps):
    idx = torch.randint(0, len(data) - seq_len - 1, (32,))
    x = torch.stack([data[i:i + seq_len] for i in idx]).to(device)
    y = torch.stack([data[i + 1:i + seq_len + 1] for i in idx]).to(device)   # target = the next character
    logits, _ = char_model(x)
    loss = F.cross_entropy(logits.reshape(-1, vocab), y.reshape(-1))
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 200 == 0 or step == steps - 1: print(f"step {step:4d}  loss {loss.item():.3f}")

In [ ]:
@torch.no_grad()
def char_generate(prompt, n=120, temperature=0.8):
    char_model.eval()
    x = torch.tensor([[stoi[c] for c in prompt]], device=device)
    logits, h = char_model(x)
    out = prompt
    for _ in range(n):
        p = F.softmax(logits[0, -1] / temperature, dim=-1)
        nxt = torch.multinomial(p, 1).item()       # sample the next character
        out += itos[nxt]
        logits, h = char_model(torch.tensor([[nxt]], device=device), h)
    return out

for t in [0.3, 0.8, 1.5]:
    print(f"--- temperature {t} ---\n{char_generate('the cat ', temperature=t)}\n")

### 2c. A real pretrained model: GPT-2 (slide 25)
GPT-2 was trained on millions of web pages. The Hugging Face `transformers` library downloads it (about 500 MB) and runs the same autoregressive loop for us.

In [ ]:
try:
    from transformers import pipeline
except ImportError:
    !pip -q install transformers
    from transformers import pipeline

try:
    generator = pipeline("text-generation", model="gpt2", device=0 if device == "cuda" else -1)
    for t in [0.2, 0.8, 1.5]:
        out = generator("Once upon a time,", max_new_tokens=40, do_sample=True,
                        temperature=t, pad_token_id=50256)
        print(f"--- temperature {t} ---\n{out[0]['generated_text']}\n")
except Exception as e:
    print("Could not download GPT-2 (internet needed). Error:", e)

---
## Load the data: handwritten digits (MNIST)
For Parts 3–6 we use **MNIST**: 60,000 images of handwritten digits, each 28 × 28 pixels in grey. If MNIST cannot be downloaded (no internet), the cell falls back to scikit-learn's small built-in digits dataset, resized to 28 × 28, so the notebook still runs.

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

def load_digits_28():
    try:
        import torchvision
        ds = torchvision.datasets.MNIST("./data", train=True, download=True)
        X = ds.data.float().unsqueeze(1) / 255.0
        y = ds.targets
        print("Loaded MNIST:", tuple(X.shape))
    except Exception as e:
        print("MNIST download failed, using scikit-learn digits instead.", type(e).__name__)
        from sklearn.datasets import load_digits
        d = load_digits()
        X = torch.tensor(d.images, dtype=torch.float32).unsqueeze(1) / 16.0
        X = F.interpolate(X, size=28, mode="bilinear", align_corners=False).clamp(0, 1)
        y = torch.tensor(d.target)
        print("Loaded sklearn digits (resized):", tuple(X.shape))
    return X, y

X, y = load_digits_28()
if QUICK:
    X, y = X[:2000], y[:2000]
loader = DataLoader(TensorDataset(X, y), batch_size=128, shuffle=True, drop_last=True)
show_grid(X[:16], title="Real training images")

---
## Part 3. Autoencoder: squeeze, then rebuild
*(Slide 13)*

* **Encoder:** 784 pixels → a tiny code `z` of just **2 numbers** (the *bottleneck*).
* **Decoder:** 2 numbers → 784 pixels.
* **Training goal:** make the output look like the input.

Because 2 numbers are so few, the network must keep only the *essence* of each digit.

In [ ]:
LATENT = 2

class Encoder(nn.Module):
    def __init__(self, out_dim):
        super().__init__()
        self.net = nn.Sequential(nn.Flatten(), nn.Linear(784, 512), nn.ReLU(),
                                 nn.Linear(512, 256), nn.ReLU(), nn.Linear(256, out_dim))
    def forward(self, x): return self.net(x)

class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(LATENT, 256), nn.ReLU(), nn.Linear(256, 512), nn.ReLU(),
                                 nn.Linear(512, 784), nn.Sigmoid())
    def forward(self, z): return self.net(z).view(-1, 1, 28, 28)

set_seed()
ae_enc, ae_dec = Encoder(LATENT).to(device), Decoder().to(device)
opt = torch.optim.Adam(list(ae_enc.parameters()) + list(ae_dec.parameters()), lr=1e-3)
EPOCHS_AE = 2 if QUICK else 15

for epoch in range(EPOCHS_AE):
    total = 0
    for xb, _ in loader:
        xb = xb.to(device)
        recon = ae_dec(ae_enc(xb))
        loss = F.binary_cross_entropy(recon, xb)      # reconstruction error
        opt.zero_grad(); loss.backward(); opt.step()
        total += loss.item()
    print(f"epoch {epoch+1}/{EPOCHS_AE}  reconstruction loss {total/len(loader):.4f}")

In [ ]:
with torch.no_grad():
    xb = X[:8].to(device)
    recon = ae_dec(ae_enc(xb))
show_grid(torch.cat([xb, recon]), nrow=8, title="Top: input   |   Bottom: rebuilt from only 2 numbers")

def plot_latent(encode_fn, title):
    with torch.no_grad():
        Z = encode_fn(X[:5000].to(device)).cpu()
    plt.figure(figsize=(6, 5))
    sc = plt.scatter(Z[:, 0], Z[:, 1], c=y[:5000], cmap="tab10", s=4)
    plt.colorbar(sc, label="digit"); plt.title(title); plt.xlabel("z1"); plt.ylabel("z2"); plt.show()
    return Z

Z_ae = plot_latent(ae_enc, "Autoencoder latent space (each dot = one image)")

**The problem with a plain autoencoder:** the map has gaps and an odd shape. If we pick random points and decode them, many come out as blurry nonsense, because the decoder never learned what lives in the gaps.

In [ ]:
with torch.no_grad():
    lo, hi = Z_ae.min(0).values, Z_ae.max(0).values
    z_rand = (lo + (hi - lo) * torch.rand(16, 2)).to(device)
    show_grid(ae_dec(z_rand), title="Autoencoder: decoding random points (often garbage)")

---
## Part 4. Variational Autoencoder (VAE): a tidy map of ideas
*(Slides 14–15)*

Two changes turn the autoencoder into a real generator:
1. The encoder outputs a **fuzzy region**: a mean `μ` and a spread `σ`. We sample `z = μ + σ × noise` (the *reparameterisation trick*).
2. The loss gets a second part, the **KL divergence**, that pulls every region towards a standard bell curve centred at 0. This keeps the map tidy and gap-free.

`loss = reconstruction error + KL divergence`

In [ ]:
class VAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = Encoder(2 * LATENT)                # outputs mu AND log-variance
        self.dec = Decoder()
    def encode(self, x):
        h = self.enc(x)
        return h[:, :LATENT], h[:, LATENT:]           # mu, logvar
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = mu + torch.exp(0.5 * logvar) * torch.randn_like(mu)   # sample inside the fuzzy region
        return self.dec(z), mu, logvar

def vae_loss(recon, x, mu, logvar):
    rec = F.binary_cross_entropy(recon, x, reduction="sum")
    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return (rec + kl) / x.shape[0], rec / x.shape[0], kl / x.shape[0]

set_seed()
vae = VAE().to(device)
opt = torch.optim.Adam(vae.parameters(), lr=1e-3)
EPOCHS_VAE = 2 if QUICK else 20

for epoch in range(EPOCHS_VAE):
    tl = tr = tk = 0
    for xb, _ in loader:
        xb = xb.to(device)
        recon, mu, logvar = vae(xb)
        loss, rec, kl = vae_loss(recon, xb, mu, logvar)
        opt.zero_grad(); loss.backward(); opt.step()
        tl += loss.item(); tr += rec.item(); tk += kl.item()
    n = len(loader)
    print(f"epoch {epoch+1}/{EPOCHS_VAE}  total {tl/n:.1f} = rebuild {tr/n:.1f} + KL {tk/n:.1f}")

In [ ]:
_ = plot_latent(lambda x: vae.encode(x)[0], "VAE latent space: centred, round and gap-free")

# Decode a whole grid of points: the 'map of ideas' made visible
n = 15
grid = torch.linspace(-2.5, 2.5, n)
with torch.no_grad():
    zs = torch.stack([torch.tensor([gx, -gy]) for gy in grid for gx in grid]).to(device)
    imgs = vae.dec(zs).cpu()
canvas = imgs.view(n, n, 28, 28).permute(0, 2, 1, 3).reshape(n * 28, n * 28)
plt.figure(figsize=(7, 7)); plt.imshow(canvas, cmap="gray"); plt.axis("off")
plt.title("Every point of the VAE map decodes to a sensible digit"); plt.show()

### Generating and morphing (slide 15)
* **Generate:** pick random `z` from a bell curve → decoder → a new digit. The encoder is not needed.
* **Morph:** take the codes of two real digits, walk in a straight line between them, and decode each step.

In [ ]:
with torch.no_grad():
    z = torch.randn(32, LATENT, device=device)       # random points from N(0, 1)
    show_grid(vae.dec(z), title="Brand-new digits sampled from the VAE")

    a, b = X[y == 0][0:1].to(device), X[y == 1][0:1].to(device)
    za, zb = vae.encode(a)[0], vae.encode(b)[0]
    steps = torch.linspace(0, 1, 10, device=device).unsqueeze(1)
    show_grid(vae.dec(za + steps * (zb - za)), nrow=10, title="Morphing a 0 into a 1 through latent space")

**Try it:** set `LATENT = 16` and re-run Parts 3–4. Images get sharper (more numbers to describe a digit), but you can no longer draw the map in 2-D. This is the classic trade-off between compression and quality.

---
## Part 5. GAN: the forger vs. the detective
*(Slides 16–18)*

* **Generator (forger):** turns random noise into fake data.
* **Discriminator (detective):** says *real* or *fake*.

Each training round (slide 17): make fakes → mix with real data → update the detective → update the forger to fool the detective → repeat.

### 5a. A toy GAN on a 2-D ring (fast, runs on any CPU)
The "real data" are points on a ring. Watch the red generated points move from a blob onto the ring.

In [ ]:
def real_ring(n):
    t = torch.rand(n) * 2 * math.pi
    return torch.stack([torch.cos(t), torch.sin(t)], 1) * 2 + 0.1 * torch.randn(n, 2)

def mlp(i, o, h=128, out_act=None):
    layers = [nn.Linear(i, h), nn.LeakyReLU(0.2), nn.Linear(h, h), nn.LeakyReLU(0.2), nn.Linear(h, o)]
    if out_act: layers.append(out_act)
    return nn.Sequential(*layers)

set_seed()
G_toy, D_toy = mlp(2, 2).to(device), mlp(2, 1).to(device)
opt_g = torch.optim.Adam(G_toy.parameters(), lr=1e-3, betas=(0.5, 0.999))
opt_d = torch.optim.Adam(D_toy.parameters(), lr=1e-3, betas=(0.5, 0.999))
bce = nn.BCEWithLogitsLoss()

STEPS_TOY = 800 if QUICK else 4000
snap_at = [0, STEPS_TOY // 8, STEPS_TOY // 2, STEPS_TOY - 1]; snaps = {}
fixed_noise = torch.randn(400, 2, device=device)
for step in range(STEPS_TOY):
    real = real_ring(256).to(device)
    fake = G_toy(torch.randn(256, 2, device=device))
    # 1) train the detective: real -> 1, fake -> 0
    d_loss = bce(D_toy(real), torch.ones(256, 1, device=device)) + bce(D_toy(fake.detach()), torch.zeros(256, 1, device=device))
    opt_d.zero_grad(); d_loss.backward(); opt_d.step()
    # 2) train the forger: wants the detective to say 1 for its fakes
    g_loss = bce(D_toy(fake), torch.ones(256, 1, device=device))
    opt_g.zero_grad(); g_loss.backward(); opt_g.step()
    if step in snap_at:
        with torch.no_grad(): snaps[step] = G_toy(fixed_noise).cpu()

ring = real_ring(500)
fig, axes = plt.subplots(1, 4, figsize=(14, 3.6))
for ax, (s, pts) in zip(axes, snaps.items()):
    ax.scatter(ring[:, 0], ring[:, 1], s=4, alpha=0.4, label="real")
    ax.scatter(pts[:, 0], pts[:, 1], s=4, c="red", label="generated")
    ax.set_title(f"step {s}"); ax.set_xlim(-3, 3); ax.set_ylim(-3, 3); ax.set_aspect("equal")
axes[0].legend(loc="lower left"); plt.show()

### 5b. A GAN that draws digits
Same game, bigger networks. GANs like data scaled to **[-1, 1]**, so the generator ends with `tanh`.

In [ ]:
NOISE = 64
G = nn.Sequential(nn.Linear(NOISE, 256), nn.LeakyReLU(0.2), nn.Linear(256, 512), nn.LeakyReLU(0.2),
                  nn.Linear(512, 784), nn.Tanh()).to(device)
D = nn.Sequential(nn.Flatten(), nn.Linear(784, 512), nn.LeakyReLU(0.2), nn.Dropout(0.3),
                  nn.Linear(512, 256), nn.LeakyReLU(0.2), nn.Dropout(0.3), nn.Linear(256, 1)).to(device)
set_seed()
opt_g = torch.optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
opt_d = torch.optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))
EPOCHS_GAN = 3 if QUICK else 40
fixed_z = torch.randn(32, NOISE, device=device)
g_hist, d_hist = [], []

for epoch in range(EPOCHS_GAN):
    for xb, _ in loader:
        real = xb.to(device) * 2 - 1                          # [0,1] -> [-1,1]
        n = real.shape[0]
        fake = G(torch.randn(n, NOISE, device=device)).view(-1, 1, 28, 28)
        ones, zeros = torch.full((n, 1), 0.9, device=device), torch.zeros(n, 1, device=device)  # 0.9 = 'label smoothing'
        d_loss = bce(D(real), ones) + bce(D(fake.detach()), zeros)
        opt_d.zero_grad(); d_loss.backward(); opt_d.step()
        g_loss = bce(D(fake), torch.ones(n, 1, device=device))
        opt_g.zero_grad(); g_loss.backward(); opt_g.step()
    g_hist.append(g_loss.item()); d_hist.append(d_loss.item())
    print(f"epoch {epoch+1}/{EPOCHS_GAN}  detective loss {d_loss.item():.3f}  forger loss {g_loss.item():.3f}")
    if epoch in (0, EPOCHS_GAN // 2, EPOCHS_GAN - 1):
        with torch.no_grad():
            show_grid((G(fixed_z).view(-1, 1, 28, 28) + 1) / 2, title=f"GAN samples after epoch {epoch+1}")

In [ ]:
plt.figure(figsize=(6, 3))
plt.plot(d_hist, label="detective (D)"); plt.plot(g_hist, label="forger (G)")
plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend(); plt.title("The two players push against each other"); plt.show()

Notice the losses do **not** steadily go down as in normal training: the two networks are competing. If the generated digits all look alike, you are seeing **mode collapse** (slide 18).

---
## Part 6. Diffusion models: create by removing noise
*(Slides 19–20)*

* **Forward process (no learning):** add a little noise at each of `T` steps until the image is pure static.
* **Reverse process (learned):** a network looks at a noisy image and the step number and **predicts the noise** that was added. Removing it step by step turns static into a picture.

A handy formula lets us jump straight to any step `t`:
`x_t = √ᾱ_t · x_0 + √(1 − ᾱ_t) · noise`, where `ᾱ_t` shrinks from 1 to almost 0.

In [ ]:
T = 400
betas = torch.linspace(1e-4, 0.03, T)          # how much noise each step adds
alphas = 1 - betas
alpha_bar = torch.cumprod(alphas, 0)           # 'how much of the original image is left'

def add_noise(x0, t, noise):
    ab = alpha_bar.to(x0.device)[t].view(-1, *[1] * (x0.dim() - 1))
    return ab.sqrt() * x0 + (1 - ab).sqrt() * noise

x0 = X[:1] * 2 - 1                             # scale to [-1, 1]
ts = [0, 25, 75, 150, 250, T - 1]
noisy = torch.cat([add_noise(x0, torch.tensor([t]), torch.randn_like(x0)) for t in ts])
show_grid((noisy + 1) / 2, nrow=6, size=1.6, title="Forward process: " + "  ".join(f"t={t}" for t in ts))

### 6a. Toy diffusion on the 2-D ring (fast)
The denoiser is a small network that receives a noisy point and the step `t`, and predicts the noise.

In [ ]:
class ToyDenoiser(nn.Module):
    def __init__(self, h=128):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(3, h), nn.SiLU(), nn.Linear(h, h), nn.SiLU(),
                                 nn.Linear(h, h), nn.SiLU(), nn.Linear(h, 2))
    def forward(self, x, t):
        return self.net(torch.cat([x, (t.float() / T).unsqueeze(1)], 1))

set_seed()
toy_eps = ToyDenoiser().to(device)
opt = torch.optim.Adam(toy_eps.parameters(), lr=1e-3)
for step in range(1500 if QUICK else 6000):
    x0 = real_ring(512).to(device)
    t = torch.randint(0, T, (512,), device=device)
    noise = torch.randn_like(x0)
    loss = F.mse_loss(toy_eps(add_noise(x0, t, noise), t), noise)   # learn to predict the noise
    opt.zero_grad(); loss.backward(); opt.step()
print("final loss:", round(loss.item(), 4))

@torch.no_grad()
def reverse_process(model, shape, keep=()):
    x = torch.randn(shape, device=device)                 # start from pure noise
    saved = {}
    for t in reversed(range(T)):
        tt = torch.full((shape[0],), t, device=device, dtype=torch.long)
        eps = model(x, tt)
        x = (x - betas[t] / (1 - alpha_bar[t]).sqrt() * eps) / alphas[t].sqrt()   # remove predicted noise
        if t > 0: x = x + betas[t].sqrt() * torch.randn_like(x)                # a little fresh randomness
        if t in keep: saved[t] = x.cpu().clone()
    return x, saved

_, saved = reverse_process(toy_eps, (600, 2), keep=(T - 1, 200, 50, 0))
fig, axes = plt.subplots(1, 4, figsize=(14, 3.6))
for ax, t in zip(axes, [T - 1, 200, 50, 0]):
    ax.scatter(saved[t][:, 0], saved[t][:, 1], s=4, c="purple")
    ax.set_title(f"reverse step t={t}"); ax.set_xlim(-3.5, 3.5); ax.set_ylim(-3.5, 3.5); ax.set_aspect("equal")
plt.suptitle("Static becomes a ring, step by step"); plt.show()

### 6b. A small diffusion model that draws digits
For images the denoiser is a small **U-Net**: it shrinks the image to see the big picture, then grows it back, with shortcut connections to keep the details. It is also told the step `t` through a *time embedding*.

⏱️ With a Colab GPU this takes about 5–10 minutes. On a CPU, set `QUICK = True` or be patient.

In [ ]:
class TimeEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__(); self.dim = dim
        self.mlp = nn.Sequential(nn.Linear(dim, dim * 4), nn.SiLU(), nn.Linear(dim * 4, dim * 4))
    def forward(self, t):                                  # sinusoidal 'clock' for the step number
        half = self.dim // 2
        f = torch.exp(-math.log(10000) * torch.arange(half, device=t.device) / half)
        a = t.float().unsqueeze(1) * f.unsqueeze(0)
        return self.mlp(torch.cat([a.sin(), a.cos()], 1))

class Block(nn.Module):
    def __init__(self, cin, cout, tdim):
        super().__init__()
        self.c1, self.c2 = nn.Conv2d(cin, cout, 3, padding=1), nn.Conv2d(cout, cout, 3, padding=1)
        self.n1, self.n2 = nn.GroupNorm(8, cout), nn.GroupNorm(8, cout)
        self.t = nn.Linear(tdim, cout)
    def forward(self, x, temb):
        h = F.silu(self.n1(self.c1(x)))
        h = h + self.t(temb)[:, :, None, None]            # tell the block which step we are on
        return F.silu(self.n2(self.c2(h)))

class TinyUNet(nn.Module):
    def __init__(self, c=32):
        super().__init__()
        self.temb = TimeEmbedding(c); td = c * 4
        self.inc = nn.Conv2d(1, c, 3, padding=1)
        self.d1, self.d2 = Block(c, c, td), Block(c, 2 * c, td)          # 28x28 -> 14x14
        self.mid = Block(2 * c, 4 * c, td)                               # 7x7
        self.u2, self.u1 = Block(4 * c + 2 * c, 2 * c, td), Block(2 * c + c, c, td)
        self.out = nn.Conv2d(c, 1, 1)
    def forward(self, x, t):
        te = self.temb(t)
        h1 = self.d1(self.inc(x), te)                                    # 28
        h2 = self.d2(F.avg_pool2d(h1, 2), te)                            # 14
        m = self.mid(F.avg_pool2d(h2, 2), te)                            # 7
        u = self.u2(torch.cat([F.interpolate(m, scale_factor=2), h2], 1), te)   # back to 14
        u = self.u1(torch.cat([F.interpolate(u, scale_factor=2), h1], 1), te)   # back to 28
        return self.out(u)

set_seed()
unet = TinyUNet().to(device)
print(f"U-Net parameters: {sum(p.numel() for p in unet.parameters()):,}")
opt = torch.optim.Adam(unet.parameters(), lr=2e-3)
EPOCHS_DIFF = 1 if QUICK else 10

for epoch in range(EPOCHS_DIFF):
    t0, total = time.time(), 0
    for xb, _ in loader:
        x0 = xb.to(device) * 2 - 1
        t = torch.randint(0, T, (x0.shape[0],), device=device)
        noise = torch.randn_like(x0)
        loss = F.mse_loss(unet(add_noise(x0, t, noise), t), noise)
        opt.zero_grad(); loss.backward(); opt.step()
        total += loss.item()
    print(f"epoch {epoch+1}/{EPOCHS_DIFF}  noise-prediction loss {total/len(loader):.4f}  ({time.time()-t0:.0f}s)")

In [ ]:
set_seed(7)
keep = (T - 1, 300, 200, 100, 50, 0)
samples, saved = reverse_process(unet.eval(), (16, 1, 28, 28), keep=keep)
show_grid((samples + 1) / 2, title="New digits generated from pure noise")

# watch ONE image being born, step by step
journey = torch.cat([saved[t][:1] for t in keep])
show_grid((journey + 1) / 2, nrow=len(keep), size=1.6, title="Reverse process: " + "  ".join(f"t={t}" for t in keep))

**Try it:** train for more epochs and watch the digits sharpen. Real image models such as Stable Diffusion use the same loop with a much bigger U-Net (or Transformer), about 1,000 noise levels and a text prompt that guides every step.

---
## Part 7 (bonus). Text-to-image with Stable Diffusion
*(Slide 20)*  Needs a **GPU runtime** and downloads about 2 GB. The pipeline contains all the parts from the slide: text encoder → U-Net denoiser loop → VAE decoder.

In [ ]:
if device != "cuda":
    print("Skipping: switch to a GPU runtime (Runtime -> Change runtime type -> T4 GPU) to run this cell.")
else:
    try:
        from diffusers import StableDiffusionPipeline
    except ImportError:
        !pip -q install diffusers transformers accelerate
        from diffusers import StableDiffusionPipeline
    pipe = StableDiffusionPipeline.from_pretrained("stable-diffusion-v1-5/stable-diffusion-v1-5",
                                                   torch_dtype=torch.float16).to("cuda")
    prompt = "a cat astronaut on the moon, digital art"
    images = []
    for seed in [1, 2, 3]:                            # same prompt + new noise = new picture
        g = torch.Generator("cuda").manual_seed(seed)
        images.append(pipe(prompt, num_inference_steps=30, guidance_scale=7.5, generator=g).images[0])
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    for ax, im, s in zip(axes, images, [1, 2, 3]):
        ax.imshow(im); ax.axis("off"); ax.set_title(f"seed {s}")
    plt.suptitle(prompt); plt.show()

---
## Summary and exercises

| Model | What it learned here | Key line of code |
|---|---|---|
| Autoregressive | next word / character | `torch.multinomial(softmax(logits / temperature))` |
| Autoencoder | 784 → 2 → 784 | `loss = BCE(recon, x)` |
| VAE | a tidy 2-D map | `z = mu + sigma * noise`, `loss = rebuild + KL` |
| GAN | forger vs. detective | alternate `d_loss` and `g_loss` updates |
| Diffusion | predict the added noise | `loss = MSE(model(x_t, t), noise)` |

**Exercises**
1. In Part 2a, add your own sentences to the corpus. How do the probabilities after "on the" change?
2. In Part 4, generate only "7"-like digits: find where 7s sit on the latent map and sample `z` near there.
3. In Part 5b, remove the label smoothing (`0.9` → `1.0`) and the `Dropout`. Is training less stable?
4. In Part 6, change `T` to 100. Are the samples faster to make? Are they worse?
5. Which model was fastest at generating? Which gave the sharpest digits? Relate this to the comparison table on slide 23.